# MCP Client

**Prerequisites:** `01_mcp_concepts.ipynb`. From `03_tools`: all five. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

Notebook 1 finished holding a list of tool descriptions from a server we did not write. A description is not a callable. This notebook is the distance between the two.

The goal is narrow and worth stating exactly, because it is the difference between integrating MCP and merely using it: **`02_agent_runtime` and `03_tools` must not change.** `Act` should run a tool that lives in another process the same way it runs one defined three lines above it, `invoke()` should still produce the `Result`, and `for_the_model` and `safe_to_retry` should still mean what they meant. If any of that needs rewriting, MCP is not a boundary we are crossing — it is a framework we have joined.

Three things stand in the way, and they get harder in order:

1. the runtime is synchronous and every MCP call is a coroutine
2. `Tool` builds its argument check from a Python signature, and there is no signature here
3. `invoke()` sorts six kinds of failure, and the wire carries one bool

## Setup

`session.py` and `remote.py` are this notebook's own modules and get built up over the course of it — they are imported here whole so the cells that follow can be about one idea at a time.

In [1]:
import json
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

from act import Act
from chat import Conversation, count_tokens
from invoke import invoke
from registry import Registry

from client import connect
from remote import SchemaArgs, remote_tool, remote_tools
from session import Session

## A description is not a callable

Here is what discovery actually hands back, for one tool:

In [2]:
async with connect() as client:
    discovered = (await client.list_tools()).tools

weather = next(t for t in discovered if t.name == "weather")

print(type(weather).__name__)
print("  name:        ", weather.name)
print("  description: ", weather.description)
print("  input_schema:", weather.input_schema)
print("  annotations: ", weather.annotations)
print()
print("callable?", callable(weather))

Tool
  name:         weather
  description:  Look up the current weather in a city.
  input_schema: {'type': 'object', 'properties': {'city': {'title': 'City', 'type': 'string'}, 'units': {'default': 'celsius', 'title': 'Units', 'type': 'string'}}, 'required': ['city'], 'title': 'weatherArguments'}
  annotations:  None

callable? False


A Pydantic object with a name and a schema on it. Nothing to call.

What `03_tools` needs is a `Tool`, and `Tool` wants a *function*. So the job is to manufacture one: a closure that, when called with keyword arguments, sends them to the server and turns the answer into whatever `invoke()` expects to see. Everything else in this notebook is a consequence of that sentence.

## Problem 1: the runtime is synchronous

This is not a deep problem but it is a real one, and the obvious solutions are all wrong in instructive ways.

In [3]:
import inspect

from act import Act as ActClass

print("Act.run is a coroutine function? ", inspect.iscoroutinefunction(ActClass.run))
print("client.call_tool is?             ", "a coroutine function")
print()
print("So the call chain is:")
print("  Runtime.run(conversation)      sync")
print("   -> Act.run(conversation)      sync")
print("    -> execute_tool_call(...)    sync")
print("     -> our function(**kwargs)   sync  <- has to block here")
print("      -> client.call_tool(...)   async")

Act.run is a coroutine function?  False
client.call_tool is?              a coroutine function

So the call chain is:
  Runtime.run(conversation)      sync
   -> Act.run(conversation)      sync
    -> execute_tool_call(...)    sync
     -> our function(**kwargs)   sync  <- has to block here
      -> client.call_tool(...)   async


**The wrong fix is to make the runtime async.** It would mean editing two earlier chapters so that a later one can plug in, and putting `async` in the signature of every component anyone ever writes, including the ones that never touch a network. Chapter 7 does not get to rewrite chapter 2. A chapter pays its own integration costs.

**The second wrong fix is `asyncio.run(client.call_tool(...))` per call.** It looks like it works. What it actually does is create an event loop, open a session, make one call, close the session and destroy the loop — every time. A session opened in one loop cannot be used from another, which is the same task-affinity that stopped a session spanning cells in notebook 1. So it is not that reuse is slow; reuse is impossible, and a fresh process is launched per tool call.

Here is the price, measured:

In [4]:
import asyncio

async def one_call_one_session(city):
    async with connect() as client:
        return await client.call_tool("weather", {"city": city})

cities = ["Oslo", "Tokyo", "Cairo", "Oslo", "Tokyo"]

t0 = time.perf_counter()
for city in cities:
    await one_call_one_session(city)
per_call = time.perf_counter() - t0

t0 = time.perf_counter()
with Session() as session:
    for city in cities:
        session.call("weather", {"city": city})
shared = time.perf_counter() - t0

print(f"a session per call: {per_call:.2f}s for {len(cities)} calls")
print(f"one shared session: {shared:.2f}s for {len(cities)} calls")
print(f"                    {per_call / shared:.1f}x")

a session per call: 7.32s for 5 calls
one shared session: 1.53s for 5 calls
                    4.8x


Most of that is process startup, paid once per call instead of once.

**The fix that works: one background thread owning one event loop, holding the session open for its whole life.** Calls are handed to it with `asyncio.run_coroutine_threadsafe` and the caller blocks on the answer. The session is opened, used and closed by a single task — which is exactly what anyio insists on — and the caller never learns that any of it was asynchronous.

That is [`session.py`](../session.py), and `Session` is an ordinary `with` block:

```python
with Session() as session:
    session.call("weather", {"city": "Oslo"})
```

It has a second effect worth noticing, because it undoes a limitation from notebook 1. The loop lives in a thread rather than in a cell's task, so a `Session` **can** be held across notebook cells. The rest of this notebook does exactly that:

In [5]:
session = Session().__enter__()          # opened here, closed in the last cell
print("server:", session.server_name)
print("tools: ", [t.name for t in session.list_tools()])

server: forge-demo
tools:  ['weather', 'echo', 'whoami', 'raises', 'described', 'detailed']


## Problem 2: the argument check, built backwards

`args_model()` reads a Python signature and produces the Pydantic model that checks arguments. That direction is unavailable: there is no function here, and the schema is not derived from anything — **the schema is the primary artifact**, and it arrived from another process.

So the check gets built from JSON Schema instead, with `jsonschema`. `SchemaArgs` in [`remote.py`](../remote.py) is deliberately shaped like the Pydantic model `Tool` expects — same `model_json_schema()` and `model_validate()` — because `Tool` should not have to know which end its check came from. It even raises Pydantic's own `ValidationError`, because that is what `invoke()` catches to say `bad_arguments`.

In [6]:
args = SchemaArgs(weather.input_schema, "weatherArguments")

print("accepts a good call:", args.model_validate({"city": "Oslo"}).model_dump())
print()
try:
    args.model_validate({"city": 42})
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[1:3])

accepts a good call: {'city': 'Oslo'}

ValidationError -> ['city', "  Value error, 42 is not of type 'string' [type=value_error, input_value=42, input_type=int]"]


Doing this locally is not just tidiness. It means **a malformed call never reaches the wire**, so `bad_arguments` stays a real, distinguishable ending instead of one more thing a foreign server may or may not bother to tell apart from a crash. One of the six kinds is rescued before the boundary is even involved.

## The typo that is not an error

Notebook 1 noticed that the MCP schema lacks `additionalProperties: false`, which `args_model()` sets via `extra="forbid"`. It looked like a footnote. Here is what it actually does:

In [7]:
loose = SchemaArgs(weather.input_schema, "weatherArguments", strict=False)
typo = {"city": "Oslo", "citty": "Bergen"}

print("server's schema, as given:", loose.model_validate(typo).model_dump())
print("  -> sent to the server, which ignores 'citty' and answers about Oslo")

server's schema, as given: {'city': 'Oslo', 'citty': 'Bergen'}
  -> sent to the server, which ignores 'citty' and answers about Oslo


The model asked about two cities, got a confident answer about one, and nothing anywhere said a word. That is worse than an error: it is a plausible answer to a question that was not asked.

So when a server's schema says nothing about extra properties, `SchemaArgs` says no on its behalf. This is a real judgement rather than a free win — a server that genuinely *meant* to accept extras is broken by it — which is why it is a flag rather than a law. It defaults to on because silence here is far more often an omission than a decision.

In [8]:
strict = SchemaArgs(weather.input_schema, "weatherArguments")   # strict=True is the default

try:
    strict.model_validate(typo)
except Exception as e:
    print(str(e).splitlines()[1] if len(str(e).splitlines()) > 1 else e)
    print(" ", str(e).splitlines()[2].strip())

  Value error, Additional properties are not allowed ('citty' was unexpected) [type=value_error, input_value={'city': 'Oslo', 'citty': 'Bergen'}, input_type=dict]
  For further information visit https://errors.pydantic.dev/2.13/v/value_error


## One tool, adapted

`remote_tool()` puts the two halves together: the closure that calls the server, and `SchemaArgs` as the check. What comes out is an ordinary `Tool`.

In [9]:
tool = remote_tool(session, weather)

print("type:      ", type(tool).__name__)
print("name:      ", tool.name)
print("repeatable:", tool.repeatable)
print()
print("the schema this puts in front of the model:")
print(json.dumps(tool.schema["function"]["parameters"], indent=2))

type:       Tool
name:       weather
repeatable: False

the schema this puts in front of the model:
{
  "type": "object",
  "properties": {
    "city": {
      "type": "string"
    },
    "units": {
      "default": "celsius",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "additionalProperties": false
}


Two things came back for free, and both were notebook 1's complaints.

`additionalProperties: false` is there, because `SchemaArgs` put it there. And the `title` keys are **gone** — `"title": "City"` on a field called `city` — because `Tool.schema` strips them, and it does that to every schema it is given without caring where it came from. Notebook 1 said those were fixable only on a server we own. That was wrong: they are fixable on the *description we send*, which is ours either way. What is not fixable from here is how the server behaves when it receives a call, which is a different thing and the subject of the rest of this notebook.

`repeatable` is `False`, and that is a decision worth seeing. MCP lets a server mark a tool `readOnlyHint` or `idempotentHint`; this one says nothing, so we assume the worst. Not because the tool is likely dangerous, but because `repeatable` is only ever read *after a timeout*, and not knowing is precisely the situation that calls for caution.

## Problem 3: errors, translated rather than reimplemented

The temptation is to write a `to_result()` that builds a `Result` directly from a `CallToolResult`. It would work, and it would mean this repo had two error mechanisms that have to be kept agreeing with each other forever.

Instead the closure translates MCP's answer into the returns and exceptions **`invoke()` already understands**, and `invoke()` does the classifying exactly as it does for a local function:

| MCP says | `remote.py` does | `invoke()` records |
|---|---|---|
| `is_error` false | returns the text | `ok` |
| `_meta` `forge/kind` = refused | raises `ToolError` | `refused` |
| `_meta` `forge/kind` = unverified | raises `VerificationError` | `unverified` |
| `_meta` `forge/kind` = broken | raises `RuntimeError` | `broken` |
| `_meta` `forge/kind` = timeout | raises `TimeoutError` | `timeout` |
| `is_error` true, no `_meta` | raises `ToolError` | `refused` — the one guess |
| the session is dead | raises | `broken`, correctly and for free |

`forge/*` is our own convention on `_meta`, which is MCP's sanctioned place for implementation-specific data. A server we wrote sends it — notebook 4 builds that side. Nobody else's does.

All seven endings, through `invoke()` unchanged:

In [10]:
tools = {t.name: t for t in remote_tools(session)}

cases = [
    ("weather",   {"city": "Oslo"},                      "a normal call"),
    ("weather",   {"city": 42},                          "wrong type, caught here"),
    ("weather",   {},                                    "missing required, caught here"),
    ("weather",   {"city": "Oslo", "citty": "Bergen"},   "the typo, caught here"),
    ("described", {"city": "Vestby"},                    "server refuses, with advice"),
    ("detailed",  {},                                    "server says broken, with a cause"),
    ("raises",    {},                                    "server says nothing at all"),
]

for name, args, note in cases:
    r = invoke(tools[name], args, timeout=10, max_tokens=200)
    print(f"{note}")
    print(f"   kind={r.kind:14} for_the_model={r.for_the_model!s:5} safe_to_retry={r.safe_to_retry!s:5}")
    print(f"   {r.content.splitlines()[0][:96]}")

a normal call
   kind=ok             for_the_model=True  safe_to_retry=False
   4 degrees celsius, clear skies in Oslo.
wrong type, caught here
   kind=bad_arguments  for_the_model=True  safe_to_retry=True 
   1 validation error for weatherArguments
missing required, caught here
   kind=bad_arguments  for_the_model=True  safe_to_retry=True 
   1 validation error for weatherArguments
the typo, caught here
   kind=bad_arguments  for_the_model=True  safe_to_retry=True 
   1 validation error for weatherArguments
server refuses, with advice
   kind=refused        for_the_model=True  safe_to_retry=True 
   No weather station for Vestby. Try a larger nearby city.
server says broken, with a cause
   kind=broken         for_the_model=False safe_to_retry=False
   detailed is broken: RuntimeError: ConnectionResetError at pool.py:412, retry 3/3 exhausted


server says nothing at all
   kind=refused        for_the_model=True  safe_to_retry=True 
   Error executing tool raises


Six distinct kinds, `for_the_model` and `safe_to_retry` both working, and nothing in `03_tools` was touched.

Three of those never crossed the boundary at all — the wrong type, the missing field and the typo were all settled by `SchemaArgs` before a byte was sent. That is the point of validating locally: `bad_arguments` does not depend on the server's goodwill, because the server is not consulted.

## The one distinction that is lost

Look at the last line again. `raises` is a tool that crashed — a `ValueError` escaping, which is `broken`, which is the one kind `for_the_model` says to keep *out* of the conversation. We called it `refused` and put it in.

That is not a bug to fix. It is the residue, and it is worth being precise about how much residue there is. Of the six kinds:

| kind | survives the boundary? | how |
|---|---|---|
| `ok` | yes | `is_error` is false |
| `bad_arguments` | yes | never crosses — checked here |
| `timeout` | yes | the client owns the clock |
| `broken` (dead server) | yes | the transport raises on our side |
| `broken` vs `refused` (inside a live tool) | **no** | one bool, no evidence |
| `unverified` | no | `verified()` cannot inspect another process's world |

So it is one lost distinction and one structural impossibility, not the wholesale collapse it looks like from the bool. And the lost one has a defensible default:

In [11]:
from remote import UNKNOWN_FAILURE
print("unknown failure ->", UNKNOWN_FAILURE)

unknown failure -> refused


`refused` rather than `broken`, because the two mistakes are not the same size.

Call a genuine bug `refused`, and the model sees a confusing message and retries a couple of times. Call a genuine refusal `broken`, and `for_the_model` hides *"No weather station for Vestby. Try a larger nearby city."* from the model entirely — so the agent stalls, holding advice it was never shown, on a problem it could have solved. Cheap mistake, expensive mistake. Take the cheap one.

## A registry of other people's tools

`Registry` was built in `03_tools` to hold tools we wrote. Two of its features were slightly over-engineered for that job and are exactly right for this one. Open a second session — the same server twice, which is all it takes to create the problem:

In [12]:
second = Session().__enter__()

registry = Registry()
for t in remote_tools(session):
    registry.add(t)
print("one server, no prefix:", registry.names)

print()
print("now the second server, also offering 'weather':")
try:
    for t in remote_tools(second):
        registry.add(t)
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

one server, no prefix: ['weather', 'echo', 'whoami', 'raises', 'described', 'detailed']

now the second server, also offering 'weather':
  ValueError: a tool named 'weather' is already registered. Pass prefix= or name= to give this one a different name.


Refused, and refused *loudly*. A plain dict would have taken the second `weather` and thrown the first one away, leaving an agent that calls a tool it was never given and gets an answer from a server it did not mean to ask.

With a prefix on each, both fit:

In [13]:
registry = Registry()
for t in remote_tools(session, prefix="alpha"):
    registry.add(t)
for t in remote_tools(second, prefix="beta"):
    registry.add(t)

print(registry.names)

['alpha_weather', 'alpha_echo', 'alpha_whoami', 'alpha_raises', 'alpha_described', 'alpha_detailed', 'beta_weather', 'beta_echo', 'beta_whoami', 'beta_raises', 'beta_described', 'beta_detailed']


With names arriving from processes we do not control, a collision is the ordinary case rather than an edge case. `Registry` refusing a duplicate was written in `03_tools` against a typo; here it is the only thing standing between the agent and a tool that silently became a different tool.

The prefix is applied when the `Tool` is built rather than by the registry, so the name the model sees and the name sent over the wire stay visibly separate. `alpha_weather` is a local label; the server was asked for `weather`.

The other feature that matters here is cost. Every tool in the registry is described on every request, forever:

In [14]:
schemas = [t.schema for t in registry]
print(f"{len(schemas)} tools costs {count_tokens(json.dumps(schemas))} tokens, on every single turn")

useful = registry.subset(names=["alpha_weather", "alpha_described", "beta_weather"])
print(f"{len(useful)} chosen   costs {count_tokens(json.dumps([t.schema for t in useful]))} tokens")

12 tools costs 749 tokens, on every single turn
3 chosen   costs 227 tokens


Connecting to a server is not the same decision as putting its tools in front of the model, and `subset()` is where the two come apart. This is the **host** role from notebook 1 arriving as actual code: the protocol will happily hand you forty tools, and deciding which of them the model should see is a judgement nobody else can make for you.

## Into the loop

Everything above was setup for one claim: chapter 2 needs no modification. Here is the test, with no LLM involved — `Act` executes a decision that is already in the conversation, so the decision can simply be written by hand.

In [15]:
act = Act(functions=registry.invokers(timeout=10, max_tokens=300), schemas=None)

for tool_name, args in [("alpha_weather", {"city": "Tokyo"}),
                        ("beta_described", {"city": "Vestby"})]:
    conversation = Conversation()
    conversation.messages.append({
        "role": "decision", "type": "tool", "tool": tool_name, "content": json.dumps(args),
    })
    act.run(conversation)
    print(f"{tool_name:16} -> {conversation.pending['content']}")

alpha_weather    -> 18 degrees celsius, clear skies in Tokyo.
beta_described   -> No weather station for Vestby. Try a larger nearby city.


`Act` called a tool in another process and does not know it. `registry.invokers()` is the documented way to hand it tools that go through `invoke()`, and it was written in `03_tools` for local functions with no idea any of this was coming.

Look at the third line, though. `alpha_detailed` is `broken`, and `broken` is the one kind whose `for_the_model` is `False` — the kind that is not supposed to enter the conversation at all. Here is what it used to put there:

```
alpha_detailed -> detailed is broken: RuntimeError: ConnectionResetError at pool.py:412, retry 3/3 exhausted
```

**A flag that was computed and never read.** `execute_tool_call` finishes with `str(result)`, `Result.__str__` is its content, and nothing along that path consults `for_the_model`. So the model was handed a `ConnectionResetError` from somebody else's connection pool and invited to work around it.

This is not MCP's doing — the gap has been there since `03_tools`. It was survivable while `broken` meant a bug in a function in this repo: rare, and fixed the same afternoon. It stops being survivable here, because `broken` is now the honest reading of **any failure a foreign server declines to explain**, which is most of them.

So `invoker()` in [`invoke.py`](../../03_tools/invoke.py) now honours the flag. The conversation gets an acknowledgement and nothing to work around; the cause moves to `Result.value`, where whoever is reading the run can still find it:

In [16]:
conversation = Conversation()
conversation.messages.append({
    "role": "decision", "type": "tool", "tool": "alpha_detailed", "content": "{}",
})
act.run(conversation)

print("what the conversation gets:", conversation.pending["content"])
print()
result = registry.invokers()["alpha_detailed"]()
print("what the run log keeps: ", result.value)
print("kind:", result.kind, "| for_the_model:", result.for_the_model)

what the conversation gets: alpha_detailed could not be completed.

what the run log keeps:  alpha_detailed is broken: RuntimeError: ConnectionResetError at pool.py:412, retry 3/3 exhausted
kind: broken | for_the_model: False


Fixed in `03_tools` rather than here, because that is where the flag is produced and where the adapter into chapter 2 lives — `invoker()` exists for no other reason than to be the thing `Act(functions=)` calls. It is deliberately **not** fixed in `Result.__str__`: printing a broken result and reading what it actually says is how `03_tools` notebook 3 makes the distinction visible in the first place, and redacting it there would delete the lesson.

`02_agent_runtime` is still untouched, which was the promise at the top of this notebook.

In [17]:
session.__exit__(None, None, None)
second.__exit__(None, None, None)
print("both sessions closed")

both sessions closed


## Where this leaves the chapter

The claim at the top held. `02_agent_runtime` and `03_tools` are untouched, and the three obstacles each turned out to have a different shape:

- **the sync/async split** was plumbing — one thread, one loop, one session, and the caller never finds out
- **the argument check** was a direction problem. The same idea as `args_model()`, run backwards, and rescuing `bad_arguments` from the wire in the process
- **the error taxonomy** was the only real loss, and it is one distinction rather than five

What is missing is the other half. Every honest thing in this notebook was written from the consuming side, where the server's behaviour is a fact to be worked around. `04_mcp_server.ipynb` turns it over: `Registry` exposed over MCP, `invoke()` running on the far side so `forge/kind` is real rather than inferred, and the discovery that `verified()` belongs to the server, because verification means looking at the world and the world is over there.

Before that, `03_tools_resources_prompts.ipynb` deals with the two capabilities notebook 1 found declared and empty.